# Exploración de datos de Instacart con Python y Pandas
**Actividad:** GA1-220501094-01-AA1-EV04 – Fundamentos en Pandas  
**Rol:** analista de datos de Instacart · **Trabajo:** individual



## Momento 1. Comprensión del problema y reconocimiento de los datos

### 1.1 Descripción del problema
El equipo de Instacart necesita saber qué información contienen sus archivos (pedidos, productos, pasillos, departamentos y artículos por pedido) y cómo consultarla para entender el comportamiento de los clientes. Mi tarea es hacer una exploración inicial: identificar la estructura, consultar registros específicos y calcular indicadores básicos.

### 1.2 Los cinco archivos
| Archivo | Qué almacena | Columnas clave |
|---|---|---|
| `instacart_orders.csv` | Un registro por **pedido**: usuario, número de pedido, día, hora y días desde el pedido anterior | `order_id`, `user_id` |
| `products.csv` | Catálogo: un registro por **producto**, con su pasillo y departamento | `product_id`, `aisle_id`, `department_id` |
| `aisles.csv` | Nombre de cada pasillo | `aisle_id` |
| `departments.csv` | Nombre de cada departamento | `department_id` |
| `order_products.csv` | Un registro por **artículo dentro de un pedido**: posición en el carrito y si es recompra | `order_id`, `product_id` |

### 1.3 Columnas para identificar pedidos, usuarios y productos
- **Pedidos:** `order_id`
- **Usuarios (clientes):** `user_id`
- **Productos:** `product_id`

### 1.4 Tres preguntas de análisis
1. ¿Cuántos pedidos se hacen los domingos y cuántos corresponden a clientes que ya han comprado más de 10 veces?
2. ¿Cuál es el promedio de días que pasan entre un pedido y el siguiente, y qué tan variable es la hora del día en que compran?
3. ¿Qué parte de los artículos pedidos son recompras (reordered = 1) y cuántos se agregan primero al carrito (add_to_cart_order = 1)?

### 1.5 ¿Por qué explorar antes de analizar?
Explorar permite conocer el tamaño, los tipos de datos, los valores ausentes y el significado de cada columna. Sin esto se pueden sacar conclusiones erróneas (por ejemplo, tratar filas como clientes, o promediar columnas con nulos sin saberlo).

## Momento 2. Importación y exploración inicial

In [4]:
import pandas as pd

# Ajusta DATA_DIR según dónde guardaste los CSV (relativo a este cuaderno)
DATA_DIR = "../data/"

orders = pd.read_csv(DATA_DIR + "instacart_orders.csv", sep=";")
products = pd.read_csv(DATA_DIR + "products.csv", sep=";")
aisles = pd.read_csv(DATA_DIR + "aisles.csv", sep=";")
departments = pd.read_csv(DATA_DIR + "departments.csv", sep=";")
order_products = pd.read_csv(DATA_DIR + "order_products.csv", sep=";")

In [5]:
dfs = {
    "orders": orders,
    "products": products,
    "aisles": aisles,
    "departments": departments,
    "order_products": order_products,
}

for nombre, df in dfs.items():
    print(f"===== {nombre}: primeros 5 =====")
    print(df.head())
    print(f"===== {nombre}: últimos 5 =====")
    print(df.tail())
    print()

===== orders: primeros 5 =====
   order_id  user_id  order_number  order_dow  order_hour_of_day  \
0   1515936   183418            11          6                 13   
1   1690866   163593             5          5                 12   
2   1454967    39980             4          5                 19   
3   1768857    82516            56          0                 20   
4   3007858   196724             2          4                 12   

   days_since_prior_order  
0                    30.0  
1                     9.0  
2                     2.0  
3                    10.0  
4                    17.0  
===== orders: últimos 5 =====
        order_id  user_id  order_number  order_dow  order_hour_of_day  \
478962   3210681     5617             5          1                 14   
478963   3270802   112087             2          3                 13   
478964    885349    82944            16          2                 11   
478965    216274     4391             3          3                  8 

In [6]:
for nombre, df in dfs.items():
    print(f"{nombre:15s} filas = {df.shape[0]:>9,}  columnas = {df.shape[1]}")

orders          filas =   478,967  columnas = 6
products        filas =    49,694  columnas = 4
aisles          filas =       134  columnas = 2
departments     filas =        21  columnas = 2
order_products  filas = 4,545,007  columnas = 4


In [7]:
# Respuestas con datos (sin escribirlas a mano)
filas = {n: d.shape[0] for n, d in dfs.items()}
cols = {n: d.shape[1] for n, d in dfs.items()}
print("Más registros:", max(filas, key=filas.get), filas[max(filas, key=filas.get)])
print("Menos columnas:", [n for n, c in cols.items() if c == min(cols.values())], min(cols.values()))

Más registros: order_products 4545007
Menos columnas: ['aisles', 'departments'] 2


### Respuestas del Momento 2

- **¿Cuál DataFrame tiene más registros?** `order_products`, con 4.545.007 filas. Es el más grande porque tiene una fila por cada artículo de cada pedido, y un mismo pedido incluye varios artículos.

- **¿Cuál tiene menos columnas?** `aisles` y `departments`, con 2 columnas cada uno: un identificador (`aisle_id` o `department_id`) y el nombre correspondiente.

- **¿Qué diferencias hay entre los datos de pedidos y los de productos?** `orders` describe la compra como evento: quién la hizo (`user_id`), cuántas compras lleva ese cliente (`order_number`), el día y la hora, y los días transcurridos desde el pedido anterior. `products` describe el producto: su nombre (`product_name`) y el pasillo y departamento al que pertenece. No se pueden unir directamente; se conectan a través de `order_products`, que tiene `order_id` (como `orders`) y `product_id` (como `products`).

- **¿Por qué es importante conocer las dimensiones de un conjunto de datos?** Porque con `shape` pude comprobar que los archivos cargaron completos: las filas coinciden con las de la guía (478.967 en `orders`, 49.694 en `products`, 134 en `aisles`, 21 en `departments` y 4.545.007 en `order_products`). También permite detectar errores de lectura, como un separador mal indicado, que mostraría una sola columna, y anticipar el tiempo y la memoria que exigirá el análisis, sobre todo con un archivo grande como `order_products`.

## Momento 3. Estructura y calidad inicial de los datos

In [8]:
for nombre, df in dfs.items():
    print(f"##### {nombre} #####")
    print("Columnas:", list(df.columns))
    print(df.dtypes)
    df.info()
    print()

##### orders #####
Columnas: ['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
order_id                    int64
user_id                     int64
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 478967 entries, 0 to 478966
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   order_id                478967 non-null  int64  
 1   user_id                 478967 non-null  int64  
 2   order_number            478967 non-null  int64  
 3   order_dow               478967 non-null  int64  
 4   order_hour_of_day       478967 non-null  int64  
 5   days_since_prior_order  450148 non-null  float64
dtypes: float64(1), int64(5)
memory usage: 21.9 MB

##### products #####
Columnas: ['product_id', 'product_name', 'aisle_i

In [9]:
# Valores ausentes por columna (solo reconocer, no corregir)
for nombre, df in dfs.items():
    print(f"--- {nombre} ---")
    print(df.isnull().sum())
    print()

--- orders ---
order_id                      0
user_id                       0
order_number                  0
order_dow                     0
order_hour_of_day             0
days_since_prior_order    28819
dtype: int64

--- products ---
product_id          0
product_name     1258
aisle_id            0
department_id       0
dtype: int64

--- aisles ---
aisle_id    0
aisle       0
dtype: int64

--- departments ---
department_id    0
department       0
dtype: int64

--- order_products ---
order_id               0
product_id             0
add_to_cart_order    836
reordered              0
dtype: int64



In [10]:
# Tabla de diagnóstico
diag = []
for nombre, df in dfs.items():
    nulos = df.isnull().sum()
    con_nulos = nulos[nulos > 0]
    obs = ", ".join(f"{c}: {n} nulos" for c, n in con_nulos.items()) or "Sin valores ausentes"
    diag.append({"archivo": nombre + ".csv", "filas": df.shape[0], "columnas": df.shape[1], "observaciones": obs})

diagnostico = pd.DataFrame(diag)
diagnostico

,archivo,filas,columnas,observaciones
0,orders.csv,478967,6,days_since_prior_order: 28819 nulos
1,products.csv,49694,4,product_name: 1258 nulos
2,aisles.csv,134,2,Sin valores ausentes
3,departments.csv,21,2,Sin valores ausentes
4,order_products.csv,4545007,4,add_to_cart_order: 836 nulos


### Respuestas del Momento 3

- **Campos que relacionan los conjuntos:** `order_id` une `orders` con `order_products`; `product_id` une `products` con `order_products`; `aisle_id` une `products` con `aisles`; y `department_id` une `products` con `departments`.

- **Diferencias entre int64, float64 y object:** `int64` guarda números enteros, como `order_id` o `order_hour_of_day`; `float64` guarda números con decimales y también aparece cuando una columna de enteros tiene valores ausentes, como `days_since_prior_order` (por eso se ve como `30.0`); y `object` (o `str`, según la versión de pandas) se usa para texto, como `product_name`.

- **Qué da `info()` que no da `shape`:** `shape` solo entrega el número de filas y columnas. `info()` además muestra el nombre y el tipo de cada columna, cuántos valores no nulos tiene y la memoria que ocupa el DataFrame.

- **Columnas con valores ausentes:** `days_since_prior_order` en `orders` (28.819 nulos de 478.967 filas), `product_name` en `products` (1.258 nulos de 49.694) y `add_to_cart_order` en `order_products` (836 nulos de 4.545.007). `aisles` y `departments` no tienen valores ausentes.

- **Dificultades de los valores ausentes:** `mean()`, `count()` y `describe()` ignoran los nulos, así que los resultados se calculan solo con una parte de las filas y no coinciden con el total de registros. Por ejemplo, el promedio de días entre pedidos se calcula con 450.148 valores y no con 478.967. Además, los nulos en `product_name` impiden saber qué producto es, y los de `add_to_cart_order` dificultan analizar el orden del carrito. En esta fase solo se identifican, no se corrigen.

## Momento 4. Indexación y filtrado lógico

### 4.1 Ejercicios individuales

**Ejercicio 1.** Seleccionar solo `product_id` y `product_name` (products)

In [11]:
ej1 = products.loc[:, ['product_id', 'product_name']]
print(ej1.head())
print(ej1.shape)

   product_id                                       product_name
0           1                         Chocolate Sandwich Cookies
1           2                                   All-Seasons Salt
2           3               Robust Golden Unsweetened Oolong Tea
3           4  Smart Ones Classic Favorites Mini Rigatoni Wit...
4           5                          Green Chile Anytime Sauce
(49694, 2)


**Ejercicio 2.** Productos con `department_id` = 1

In [12]:
ej2 = products.loc[products['department_id'] == 1]
print(ej2.head())
print('Registros:', len(ej2))

    product_id                                       product_name  aisle_id  \
3            4  Smart Ones Classic Favorites Mini Rigatoni Wit...        38   
7            8                  Cut Russet Potatoes Steam N' Mash       116   
11          12                         Chocolate Fudge Layer Cake       119   
17          18                Pizza for One Suprema  Frozen Pizza        79   
29          30         Three Cheese Ziti- Marinara with Meatballs        38   

    department_id  
3               1  
7               1  
11              1  
17              1  
29              1  
Registros: 4008


**Ejercicio 3.** Pedidos realizados a las 10 horas

In [13]:
ej3 = orders.loc[orders['order_hour_of_day'] == 10]
print(ej3.head())
print('Registros:', len(ej3))

    order_id  user_id  order_number  order_dow  order_hour_of_day  \
19    294269   107523            29          2                 10   
28    133707   182261             1          3                 10   
36   3378084   127629            21          1                 10   
49   2161799    87345             7          0                 10   
51   1739741   118405            48          2                 10   

    days_since_prior_order  
19                     1.0  
28                     NaN  
36                    20.0  
49                    24.0  
51                     3.0  
Registros: 40578


**Ejercicio 4.** Pedidos con `order_dow` = 0 (domingo)

In [14]:
ej4 = orders.loc[orders['order_dow'] == 0]
print(ej4.head())
print('Registros:', len(ej4))

    order_id  user_id  order_number  order_dow  order_hour_of_day  \
3    1768857    82516            56          0                 20   
5    1843871    20084             5          0                 11   
7    2521283    97711             2          0                 18   
23   3406282    19356            21          0                  0   
25   1895419   169793             2          0                 11   

    days_since_prior_order  
3                     10.0  
5                     15.0  
7                     22.0  
23                    14.0  
25                     8.0  
Registros: 84090


**Ejercicio 5.** Pedidos con `order_number` > 10

In [15]:
ej5 = orders.loc[orders['order_number'] > 10]
print(ej5.head())
print('Registros:', len(ej5))

    order_id  user_id  order_number  order_dow  order_hour_of_day  \
0    1515936   183418            11          6                 13   
3    1768857    82516            56          0                 20   
6    2511640   148233            40          3                  8   
8    2266494    34954            40          5                 15   
10   1162566    43738            33          4                 11   

    days_since_prior_order  
0                     30.0  
3                     10.0  
6                      2.0  
8                      4.0  
10                     2.0  
Registros: 244072


**Ejercicio 6.** Pedidos con `days_since_prior_order` > 15 (los nulos quedan fuera: `NaN > 15` es False)

In [16]:
ej6 = orders.loc[orders['days_since_prior_order'] > 15]
print(ej6.head())
print('Registros:', len(ej6))

    order_id  user_id  order_number  order_dow  order_hour_of_day  \
0    1515936   183418            11          6                 13   
4    3007858   196724             2          4                 12   
7    2521283    97711             2          0                 18   
14   2843219    74699             4          6                 11   
26   1844247   122558             4          6                 18   

    days_since_prior_order  
0                     30.0  
4                     17.0  
7                     22.0  
14                    30.0  
26                    30.0  
Registros: 110706


**Ejercicio 7.** Registros con `reordered` = 1 (order_products)

In [17]:
ej7 = order_products.loc[order_products['reordered'] == 1]
print(ej7.head())
print('Registros:', len(ej7))

   order_id  product_id  add_to_cart_order  reordered
1    567889        1560                1.0          1
2   2261212       26683                1.0          1
3    491251        8670               35.0          1
4   2571142        1940                5.0          1
5   2456893       21616                4.0          1
Registros: 2683838


**Ejercicio 8.** Productos añadidos en la primera posición del carrito

In [18]:
ej8 = order_products.loc[order_products['add_to_cart_order'] == 1]
print(ej8.head())
print('Registros:', len(ej8))

    order_id  product_id  add_to_cart_order  reordered
1     567889        1560                1.0          1
2    2261212       26683                1.0          1
14   1961225       37553                1.0          1
16    639939       10017                1.0          1
23    750040        8518                1.0          0
Registros: 450046


### 4.2 Filtros con condiciones simultáneas
Cada condición va entre paréntesis y se combina con `&` (y) o `|` (o).

In [19]:
filtro_a = orders.loc[(orders['order_hour_of_day'] == 10) & (orders['order_number'] > 10)]
print(filtro_a.head())
print('Pedidos a las 10 h y order_number > 10:', len(filtro_a))

     order_id  user_id  order_number  order_dow  order_hour_of_day  \
19     294269   107523            29          2                 10   
36    3378084   127629            21          1                 10   
51    1739741   118405            48          2                 10   
92      60724   110808            78          0                 10   
140   3156095   113920            14          2                 10   

     days_since_prior_order  
19                      1.0  
36                     20.0  
51                      3.0  
92                      2.0  
140                     2.0  
Pedidos a las 10 h y order_number > 10: 21537


In [20]:
filtro_b = products.loc[(products['department_id'] == 1) & (products['aisle_id'] > 50)]
print(filtro_b.head())
print('Productos del depto 1 con aisle_id > 50:', len(filtro_b))

     product_id                         product_name  aisle_id  department_id
7             8    Cut Russet Potatoes Steam N' Mash       116              1
11           12           Chocolate Fudge Layer Cake       119              1
17           18  Pizza for One Suprema  Frozen Pizza        79              1
45           46       Raisin Cinnamon Bagels 5 count        58              1
127         128             Organic Wild Blueberries       116              1
Productos del depto 1 con aisle_id > 50: 1618


In [21]:
# Comparación: cómo baja la cantidad de registros al sumar condiciones
print('Pedidos totales:            ', len(orders))
print('Solo hora = 10:             ', len(ej3))
print('Solo order_number > 10:     ', len(ej5))
print('Ambas condiciones (&):      ', len(filtro_a))

Pedidos totales:             478967
Solo hora = 10:              40578
Solo order_number > 10:      244072
Ambas condiciones (&):       21537


### Explicación de cada resultado

1. **Columnas `product_id` y `product_name`:** se obtuvo una tabla de 49.694 filas y 2 columnas. Se conservan todos los productos, pero solo con su identificador y su nombre.
2. **Productos del departamento 1:** hay 4.008 productos, es decir, cerca del 8,1 % del catálogo de 49.694.
3. **Pedidos realizados a las 10 horas:** son 40.578 pedidos, alrededor del 8,5 % del total de 478.967.
4. **Pedidos en domingo (`order_dow` = 0):** son 84.090 pedidos, aproximadamente el 17,6 % del total.
5. **Pedidos con `order_number` > 10:** son 244.072, un poco más de la mitad (51,0 %) de los pedidos. Es decir, la mitad de los pedidos corresponde a compras que no son de los primeros diez pedidos de un cliente.
6. **Pedidos con más de 15 días desde el anterior:** son 110.706, cerca del 23,1 % del total. Los valores ausentes de `days_since_prior_order` quedan fuera de este filtro, porque `NaN > 15` es falso.
7. **Artículos con `reordered` = 1:** son 2.683.838, cerca del 59,1 % de los 4.545.007 artículos. Más de la mitad de lo que se pide ya se había pedido antes.
8. **Productos agregados primero al carrito:** son 450.046 registros con `add_to_cart_order` = 1, aproximadamente el 9,9 % de los artículos.

**Filtros simultáneos:**
- **A las 10 horas y con `order_number` > 10:** son 21.537 pedidos, mucho menos que los 40.578 de la hora sola y que los 244.072 de `order_number` sola, porque deben cumplirse las dos condiciones a la vez. Representan cerca del 4,5 % del total de pedidos.
- **Departamento 1 con `aisle_id` > 50:** son 1.618 productos, menos que los 4.008 del departamento 1 completo, porque el filtro excluye los de pasillos con identificador 50 o menor. Equivalen a un 3,3 % del catálogo.

### Respuestas del Momento 4

- **¿Qué diferencia existe entre seleccionar columnas y filtrar filas?** Seleccionar columnas reduce el ancho de la tabla: elijo qué atributos quiero ver, como en el ejercicio 1, donde quedaron solo `product_id` y `product_name`. Filtrar filas reduce el largo: me quedo con los registros que cumplen una condición, como los 84.090 pedidos de domingo del ejercicio 4.

- **¿Qué ventajas ofrece `loc[]` al consultar información específica?** Permite indicar filas y columnas en una sola expresión, y trabajar con filtros lógicos y con nombres de columna en vez de posiciones. Con él se escribe en una línea lo que sin pandas requeriría un bucle con `if`.

- **¿Qué ocurre con la cantidad de registros cuando se aplican varias condiciones?** Disminuye. Con `&` deben cumplirse todas las condiciones a la vez. Los pedidos pasaron de 478.967 a 244.072 con `order_number` > 10, y a solo 21.537 al exigir además que fueran de las 10 horas.

- **¿Por qué es importante utilizar correctamente los operadores lógicos?** Porque cambian el resultado. `&` exige que se cumplan todas las condiciones y `|` que se cumpla al menos una, y `>` no es igual a `>=`. En pandas se escriben con paréntesis en cada condición; si se usan mal, da error o devuelve filas distintas de las que se buscaban.

## Momento 5. Indicadores básicos

In [22]:
n_pedidos = orders.shape[0]
n_productos = products.shape[0]
prom_order_number = orders['order_number'].mean()
prom_dias = orders['days_since_prior_order'].mean()   # ignora nulos
n_reordered = order_products[order_products['reordered'] == 1]['reordered'].count()
suma_reordered = order_products['reordered'].sum()

print('Registros de pedidos:      ', n_pedidos)
print('Registros de productos:    ', n_productos)
print('Promedio order_number:     ', prom_order_number)
print('Promedio days_since_prior: ', prom_dias)
print('Artículos con reordered=1: ', n_reordered)
print('Suma de reordered:         ', suma_reordered)

Registros de pedidos:       478967
Registros de productos:     49694
Promedio order_number:      17.157808366755955
Promedio days_since_prior:  11.101750979677794
Artículos con reordered=1:  2683838
Suma de reordered:          2683838


In [23]:
print(orders['order_hour_of_day'].describe())
print()
print(order_products['add_to_cart_order'].describe())

count    478967.000000
mean         13.446676
std           4.224986
min           0.000000
25%          10.000000
50%          13.000000
75%          16.000000
max          23.000000
Name: order_hour_of_day, dtype: float64

count    4.544171e+06
mean     8.351669e+00
std      7.080368e+00
min      1.000000e+00
25%      3.000000e+00
50%      6.000000e+00
75%      1.100000e+01
max      6.400000e+01
Name: add_to_cart_order, dtype: float64


In [24]:
# Para diferenciar registros, pedidos y usuarios
print('Registros en orders:        ', len(orders))
print('Pedidos distintos (order_id):', orders['order_id'].nunique())
print('Usuarios distintos (user_id):', orders['user_id'].nunique())

Registros en orders:         478967
Pedidos distintos (order_id): 478952
Usuarios distintos (user_id): 157437


> **Nota (`nunique()`):** este método no se vio en las lecciones, pero sirve para diferenciar *registros*, *pedidos* y *usuarios*, tal como pide la actividad. Si las cifras difieren entre sí, es porque hay duplicados o porque un mismo usuario hizo varios pedidos.

###  Tabla de resultados e interpretación


| Pregunta | Valor obtenido | Interpretación |
|---|---|---|
| ¿Cuántos registros de pedidos? | 478.967 | Es la cantidad de filas del archivo; no equivale al número de clientes ni de pedidos distintos |
| ¿Cuántos registros de productos? | 49.694 | Tamaño del catálogo de productos |
| Promedio de `order_number` | 17,16 | En promedio, cada pedido registrado es la compra número 17 de su cliente, lo que indica clientes con bastante historial |
| Promedio de `days_since_prior_order` | 11,10 días | Los clientes vuelven a comprar, en promedio, cada 11 días; se calcula con los 450.148 valores no nulos |
| Artículos con `reordered` = 1 | 2.683.838 | Cantidad de artículos que el cliente ya había pedido antes; es el 59,05 % de los 4.545.007 registros |
| Suma de `reordered` | 2.683.838 | Coincide con el conteo anterior porque la columna solo contiene 0 y 1, así que sumar equivale a contar los 1 |
| `describe()` de `order_hour_of_day` | media 13,45; desv. 4,22; mín. 0; 25 % 10; mediana 13; 75 % 16; máx. 23 | La mitad de los pedidos se hace entre las 10 h y las 16 h, con centro hacia las 13 h; hay pedidos a todas las horas, de 0 a 23 |
| `describe()` de `add_to_cart_order` | conteo 4.544.171; media 8,35; desv. 7,08; mín. 1; 25 % 3; mediana 6; 75 % 11; máx. 64 | Un artículo suele ocupar la posición 6 o menor, pero hay carritos de hasta 64 artículos; la media es mayor que la mediana, lo que indica pocos pedidos muy grandes. El conteo es 836 menos que el total de filas, por los valores ausentes |

**Registros, pedidos y usuarios:** la tabla `orders` tiene 478.967 registros, 478.952 `order_id` distintos y 157.437 `user_id` distintos. Por eso el número de filas no equivale al número de clientes: cada cliente hizo varios pedidos, y además hay 15 filas con un `order_id` repetido, que son duplicados.



## Momento 6. Síntesis final

### 6.1 Preguntas de análisis iniciales y respuestas
1. **¿Cuántos pedidos se hacen los domingos y cuántos son de clientes con más de 10 compras?** Hay 84.090 pedidos en domingo (cerca del 17,6 % del total) y 244.072 pedidos con `order_number` mayor que 10 (el 51,0 %).
2. **¿Cuál es el promedio de días entre pedidos y qué tan variable es la hora de compra?** El promedio es de 11,10 días. La hora tiene una desviación estándar de 4,22 y la mitad de los pedidos se hace entre las 10 h y las 16 h.
3. **¿Qué parte de los artículos son recompras y cuántos se agregan primero al carrito?** Hay 2.683.838 artículos de recompra, el 59,05 % de los 4.545.007 registros, y 450.046 artículos fueron agregados primero al carrito (alrededor del 9,9 %).



### 6.2 Principales hallazgos
- Los pedidos se concentran en horas centrales del día: la mediana de `order_hour_of_day` es 13 y la mitad de los pedidos se hace entre las 10 h y las 16 h.
- Los clientes compran con frecuencia: el promedio es de 11,10 días entre pedidos y el 59,05 % de los artículos son recompras, es decir, productos que el cliente ya había pedido antes.
- Los carritos suelen ser pequeños, con mediana de 6 en `add_to_cart_order`, pero hay pedidos muy grandes de hasta 64 artículos.
- `order_products` es la tabla más grande (4.545.007 filas) y conecta pedidos con productos; hay 157.437 usuarios distintos, mucho menos que las 478.967 filas de `orders`.



### 6.3 Cómo ayudaron la indexación y los filtros
`loc[]` y el filtrado booleano permitieron aislar subconjuntos (por hora, día, departamento, recompra) sin recorrer los datos con bucles, y combinar condiciones con `&` para consultas más precisas.

### 6.4 Limitaciones de los datos
- Valores ausentes: 28.819 en `days_since_prior_order`, 1.258 en `product_name` y 836 en `add_to_cart_order`, que hacen que promedios y conteos se calculen con menos filas que el total.
- Duplicados: `orders` tiene 478.967 filas pero solo 478.952 `order_id` distintos, es decir, 15 pedidos repetidos que no se corrigieron en esta fase.
- Número de filas no es número de clientes: hay 157.437 usuarios distintos en 478.967 registros.
- El departamento `missing` indica productos sin departamento asignado.
- El análisis es descriptivo: muestra qué ocurre, pero no explica por qué.
- `order_products` es muy grande (más de 4,5 millones de filas) y exige memoria.



### 6.5 Tres conclusiones
1. Las cinco tablas se relacionan por `order_id`, `product_id`, `aisle_id` y `department_id`, y `order_products` conecta los pedidos con los productos, lo que permite combinarlas en análisis posteriores.
2. Los clientes de Instacart compran con frecuencia y de forma repetida: el promedio entre pedidos es de 11,10 días y el 59,05 % de los artículos son recompras, lo que indica que gran parte de las compras repiten productos ya pedidos.
3. Los datos son útiles pero requieren limpieza antes de un análisis más profundo, por los 28.819 nulos en `days_since_prior_order`, los 1.258 en `product_name`, los 836 en `add_to_cart_order` y los 15 pedidos duplicados.